In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
from datetime import datetime, timezone
import sys, subprocess
root = Path('/content/drive/MyDrive/CameraTrap_Phase2')
run = root/'phase2_results/final_sprint_v1'
src = run/'code'
subprocess.run([sys.executable,'-m','pip','install','-q','-r',str(src/'requirements.txt')],check=True)
sys.path.insert(0,str(src))
import numpy as np
import torch
from sklearn.linear_model import LogisticRegression
from torch.utils.data import DataLoader
from common import context, metrics, temperature, read, save, sha
from runtime import Classifier
from train import infer, Crops, samples
_,_,rows,classes = context(root)
policy = read(run/'policy.json')
entry = policy['species_classifier']
assert sha(entry['path']) == entry['sha256']
device = 'cuda' if torch.cuda.is_available() else 'cpu'
net = Classifier(entry['path'],device).net.eval()
head = net.classifier[2]
net.classifier[2] = torch.nn.Identity()
out = run.parent/'species_camera_cv'/datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
out.mkdir(parents=True)
def extract(split):
    print('Extracting', split, flush=True)
    loader=DataLoader(Crops(samples(rows,classes,split),root),batch_size=24,num_workers=2)
    return infer(net,loader,device)
x,y=extract('train')
v,vy=extract('val_select')
layer=head[-1]
original_w=layer.weight.detach().cpu().numpy()[:14]
original_b=layer.bias.detach().cpu().numpy()[:14]
def score(w,b):
    return metrics(v@w.T+b,vy)
baseline=score(original_w,original_b)
best=baseline
chosen='original'
weights,bias=original_w,original_b
trials=[]
# Select regularization using camera-disjoint folds of the training data.
from sklearn.model_selection import GroupKFold
training = samples(rows, classes, 'train')
camera_by_id = {row['id']: row['camera'] for row in rows}
groups = np.asarray([camera_by_id[item['id']] for item in training])
assert len(groups) == len(y)
candidates = []
for balance in (None, 'balanced'):
    for strength in (0.001, 0.01, 0.1, 1.0):
        fold_scores = []
        for train_idx, valid_idx in GroupKFold(n_splits=3).split(x, y, groups):
            if len(np.unique(y[train_idx])) != 14:
                continue
            fit = LogisticRegression(C=strength, class_weight=balance, max_iter=500)
            fit.fit(x[train_idx], y[train_idx])
            prediction = fit.predict(x[valid_idx])
            from sklearn.metrics import f1_score
            labels = np.unique(y[valid_idx])
            fold_scores.append(float(f1_score(y[valid_idx], prediction, labels=labels,
                                              average='macro', zero_division=0)))
        if not fold_scores:
            continue
        candidates.append((float(np.mean(fold_scores)), balance, strength, len(fold_scores)))
        print('Camera folds', balance, strength, fold_scores, flush=True)
assert candidates, 'Camera-disjoint folds lack sufficient species coverage'
# Only the two strongest training-fold candidates enter validation selection.
for cv_score, balance, strength, fold_count in sorted(candidates, key=lambda item: item[0], reverse=True)[:2]:
    fit=LogisticRegression(C=strength,class_weight=balance,max_iter=500).fit(x,y)
    assert np.array_equal(fit.classes_,np.arange(14))
    for mix in (.5, 1.):
        w=(1-mix)*original_w+mix*fit.coef_
        b=(1-mix)*original_b+mix*fit.intercept_
        m=score(w,b)
        trials.append(dict(balance=balance,C=strength,mix=mix,camera_fold_macro_f1=cv_score,
                           camera_folds=fold_count,accuracy=m['accuracy'],macro_f1=m['macro_f1']))
        if m['accuracy'] >= baseline['accuracy'] and m['macro_f1'] > best['macro_f1']:
            best,chosen,weights,bias=m,str((balance,strength,mix)),w,b
cal,cy=extract('val_calibrate')
temp=temperature(cal@weights.T+bias,cy)
head_path=out/'species_head.npz'
np.savez(head_path,weight=weights.astype('float32'),bias=bias.astype('float32'))
save(out/'selection.json',dict(source_checkpoint=entry['sha256'],head_sha256=sha(head_path),classes=classes,
    temperature=temp,selected=chosen,baseline_validation=baseline,selected_validation=metrics(v@weights.T+bias,vy,temp),
    trials=trials,scope='Known-class crops, selection on val_select, temperature on val_calibrate'))
print('Selected:',chosen,'validation macro F1:',baseline['macro_f1'],'->',best['macro_f1'],'saved:',out)


# Historical test was previously exposed during project development.
# Freeze the chosen head before inspecting it.
save(out/'freeze.json',dict(head_sha256=sha(head_path),selection='training camera folds and val_select',
                            historical_test_independent=False))
tx,ty=extract('test')
historical=metrics(tx@weights.astype('float32').T+bias.astype('float32'),ty,temp)
save(out/'historical_species_metrics.json',dict(historical,head_sha256=sha(head_path),independent_test=False,
                                                 scope='known-class ground-truth crops'))
print('Historical macro F1:',historical['macro_f1'],'ECE:',historical['ece'])
print('Saved:',out)
